# 01 · STFT / ISTFT 交互教程

目标：亲手感受 **窗长 / 窗型 / hop** 对时频分析的影响，理解 COLA 与完美重构。
配套：`core/stft.py`、`docs/INTERVIEW-WP01-STFT.md`。

> 运行方式：仓库根目录 `conda activate avsuite`，`jupyter notebook` 打开本文件，从上到下逐格运行。
> 想动手就改参数（标注 ✏️ 的格子），重新运行看现象。

In [ ]:
import sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '..')
from core.stft import STFT
from core.istft import ISTFT
torch.manual_seed(0)
sr = 16000
# 造一个合成语音：基频 150Hz 的谐波 + 包络，模拟浊音
t = np.arange(sr) / sr
sig = sum((1.0/k) * np.sin(2*np.pi*150*k*t) for k in range(1, 8))
sig *= (0.5 + 0.5*np.sin(2*np.pi*3*t))  # 3Hz 幅度包络
wav = torch.from_numpy(sig).float().unsqueeze(0)
print('signal', wav.shape)

## 1. 窗长的时频不确定性（✏️ 改 n_fft）
短窗保瞬态（时间分辨率↑）、长窗保谐波（频率分辨率↑）——这就是时频不确定性，
也是多尺度 STFT loss（`core/losses.py`）存在的理由。

In [ ]:
def show_spec(y, n_fft, hop, ax, title):
    stft = STFT(n_fft=n_fft, hop=hop)
    S = stft.transform(y)[0].abs().numpy()
    ax.imshow(20*np.log10(S+1e-6), origin='lower', aspect='auto',
              extent=[0, len(y[0])/sr, 0, sr/2], cmap='magma', vmin=-60, vmax=20)
    ax.set_title(title); ax.set_ylabel('Hz')

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, nf in zip(axes, [128, 512, 2048]):   # ✏️ 改这三个窗长
    show_spec(wav, nf, nf//4, ax, f'n_fft={nf} ({nf/sr*1000:.0f}ms)')
plt.tight_layout(); plt.show()

## 2. 完美重构：STFT→ISTFT 误差（COLA）
hann 窗 50% 重叠满足 COLA，ISTFT 用窗平方归一（WOLA）可精确重构。
本仓 `core/stft.py` 与 `torch.stft` 对拍 <1e-10（见 `tests/test_stft.py`）。

In [ ]:
for nf, hp in [(512, 256), (512, 128), (1024, 256)]:   # ✏️ 改窗长/hop
    stft, istft = STFT(nf, hp), ISTFT(nf, hp)
    X = stft.transform(wav)
    rec = istft.inverse(X, length=wav.shape[-1])
    err = (rec - wav).abs().max().item()
    print(f'n_fft={nf} hop={hp}  重构误差 max|x-x̂| = {err:.2e}')

## 3. 面试自检（合上代码回答）
1. 写出 STFT 的数学定义式。
2. 为什么 hann 窗 50% 重叠能完美重构？（COLA 条件）
3. 为什么相位不能直接做 MSE 回归？（相位卷绕 / 2π 周期不连续）
4. 流式 STFT 的算法时延从哪来？（本仓 512/320 → 窗内前瞻 12ms）